# 15. 3Sum
**Difficulty:** 🟡 Medium · **Topic:** Array · **LeetCode:** https://leetcode.com/problems/3sum/

## 💡 Concepts

**Core concept(s):** **Sorting + two pointers**, plus **duplicate skipping**. A hash-set variant also reaches O(n²).

**Why it applies here:** Fix one number `a`; the rest becomes a 2-sum for target `-a`. On a **sorted** array, two pointers find all pairs summing to `-a` in one linear sweep, and sortedness makes it trivial to skip duplicate triples. This turns the O(n³) triple loop into O(n²).

**Key intuition / mental model:** Sort. For each `a` (left anchor), walk `lo` up and `hi` down: if the triple sum is too small move `lo` right, too big move `hi` left, equal record it — then skip equal neighbors to avoid duplicates.

---

### 📚 What is the Two-Pointer Technique?
Two indices sweeping a **sorted** array (often from both ends). The current sum tells you which pointer to move, so you scan pairs in **O(n)** with **O(1)** space instead of a nested loop.

### 📚 What is a Hash Set? (for the alternative)
A `set` gives O(1) membership. In the hash-set approach, for each anchor we scan the rest once and use a set to find the needed complement — also O(n) per anchor → O(n²) overall.

### 📚 Why sort first?
Sorting (**O(n log n)**) both enables the two-pointer sweep and makes **de-duplication** a simple "skip equal neighbors" step.

## 📝 Problem

Return all **unique** triplets `[a, b, c]` in `nums` with `a + b + c == 0`. No duplicate triplets.

**Example**
```
Input:  nums = [-1, 0, 1, 2, -1, -4]
Output: [[-1, -1, 2], [-1, 0, 1]]
```
**Constraints:** `3 <= len(nums) <= 3000`.

### Approach 1 — Brute Force (worst)

**Idea:** Try all triples `i<j<k`; collect those summing to 0 into a set (sorted tuples) to dedup.

**Time complexity:** `O(n^3)`.

**Space complexity:** `O(k)` for the result set.

In [ ]:
from typing import List

def three_sum_brute(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    res = set()                            # use a set of sorted triples to auto-dedupe
    for i in range(n):                     # try every triple i < j < k
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nums[i] + nums[j] + nums[k] == 0:
                    res.add(tuple(sorted((nums[i], nums[j], nums[k]))))  # store sorted
    return [list(t) for t in res]

### Approach 2 — Hash Set per Anchor (better)

**Idea:** Sort (for easy dedup). Fix anchor `i`; scan the rest with a set, looking for `need = -nums[i]-nums[j]` seen earlier.

**Time complexity:** `O(n^2)`.

**Space complexity:** `O(n)` for the per-anchor set.

In [ ]:
from typing import List

def three_sum_hashset(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)                    # sort so duplicates are adjacent (easy to skip)
    n = len(nums)
    res = set()
    for i in range(n):                     # fix the first number
        if i > 0 and nums[i] == nums[i - 1]:
            continue                       # skip duplicate anchors
        seen = set()                       # values seen while scanning the rest
        for j in range(i + 1, n):
            need = -nums[i] - nums[j]       # the third number that makes the sum 0
            if need in seen:               # did we already pass that third number?
                res.add((nums[i], need, nums[j]))  # record the triple (already ascending)
            seen.add(nums[j])
    return [list(t) for t in res]

### Approach 3 — Sort + Two Pointers (optimal)

**Idea:** Sort. For each anchor `i`, use `lo`/`hi` pointers to find all pairs summing to `-nums[i]`, skipping duplicate neighbors. Early-break once `nums[i] > 0` (no positive anchor can start a zero-sum triple).

**Time complexity:** `O(n^2)` (sort is O(n log n)).

**Space complexity:** `O(1)` extra (besides output).

In [ ]:
from typing import List

def three_sum_optimal(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)                    # sorting enables two pointers + easy dedup
    n = len(nums)
    res = []
    for i in range(n - 2):                 # fix the first (smallest) number
        if nums[i] > 0:
            break                          # smallest is positive -> no way to reach 0
        if i > 0 and nums[i] == nums[i - 1]:
            continue                       # skip duplicate anchors
        lo, hi = i + 1, n - 1              # two pointers over the rest
        while lo < hi:
            s = nums[i] + nums[lo] + nums[hi]
            if s < 0:
                lo += 1                    # sum too small -> need a bigger value
            elif s > 0:
                hi -= 1                    # sum too big -> need a smaller value
            else:
                res.append([nums[i], nums[lo], nums[hi]])  # found a triple
                lo += 1; hi -= 1
                while lo < hi and nums[lo] == nums[lo - 1]:
                    lo += 1                # skip duplicate second numbers
                while lo < hi and nums[hi] == nums[hi + 1]:
                    hi -= 1                # skip duplicate third numbers
    return res

In [ ]:
# Correctness check (compare as sets of sorted tuples)
def norm(triples):
    return {tuple(sorted(t)) for t in triples}

tests = [
    ([-1, 0, 1, 2, -1, -4], {(-1, -1, 2), (-1, 0, 1)}),
    ([0, 0, 0, 0], {(0, 0, 0)}),
    ([1, 2, -2, -1], set()),
]
for nums, expected in tests:
    b = norm(three_sum_brute(nums))
    h = norm(three_sum_hashset(nums))
    o = norm(three_sum_optimal(nums))
    print(f"{nums} -> #brute={len(b)}, #hashset={len(h)}, #optimal={len(o)} | expected {len(expected)}")
    assert b == h == o == expected, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit) so the measurement reflects the true bound. Sub-millisecond rows are noisy — look at the trend, not one number.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    # all positive & distinct -> no zero-sum triple; brute still does full C(n,3) work
    nums = list(range(1, n + 1))
    return (nums,)

solutions = {
    "brute   O(n^3)": three_sum_brute,
    "hashset O(n^2)": three_sum_hashset,
    "optimal O(n^2)": three_sum_optimal,
}
sizes = [50, 100, 200, 400]                # kept small: brute is cubic

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Reduce k-Sum by one dimension:** Fix an anchor and solve the remaining (k-1)-Sum. 3Sum = anchor + 2Sum; the idea generalizes to 4Sum, kSum.
- **Sort + two pointers:** On sorted data, two pointers find all pairs for a target in O(n) and make dedup a neighbor-skip.
- **Signal to reach for it:** "triplets/pairs summing to X", "no duplicate combinations", "k numbers that add to a target".
- **Related problems:** Two Sum II, 4Sum, 3Sum Closest, 3Sum Smaller.
- **Common pitfalls:** (1) emitting duplicate triplets — skip equal neighbors for anchor **and** both pointers; (2) moving only one pointer after a hit; (3) forgetting the `nums[i] > 0` early break that saves time.